# Week 3 Three-Model RAG Blind Evaluation

This notebook loads the frozen aggregate only; it does not rerun retrieval, generation, or a Judge. `W03-BLIND-FARI-001` was inspected during adapter qualification and is excluded from the seven-question uninspected aggregate. Automatic RAGAS metrics remain diagnostic because the local Judge is uncalibrated; a separate AI qualitative calibration reviews all eight Llama RAG answers.

In [1]:
import json
from pathlib import Path

summary_path = Path('W03_RAG_Three_Model_Blind_Summary.json')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
assert summary['shared_input_audit']['passed']
assert summary['excluded_eval_ids'] == ['W03-BLIND-FARI-001']
print('Shared input audit: PASS')
print('Models:', ', '.join(summary['models']))
print('Uninspected evaluation items per model: 7')

Shared input audit: PASS
Models: flan_t5_base, mistral_7b_instruct_v0_2, llama31_8b_instruct
Uninspected evaluation items per model: 7


## Base versus RAG

In [2]:
print(f"{'Model':32} {'Base rel.':>10} {'RAG rel.':>10} {'Delta':>10} {'Faithful':>10}")
for model, data in summary['models'].items():
    metrics = data['ragas_provisional']
    base = metrics['base']['answer_relevance']['mean']
    rag = metrics['rag']['answer_relevance']['mean']
    faith = metrics['rag']['faithfulness_to_retrieved_context']['mean']
    print(f"{model:32} {base:10.6f} {rag:10.6f} {rag-base:+10.6f} {faith:10.6f}")

Model                             Base rel.   RAG rel.      Delta   Faithful
flan_t5_base                       0.223139   0.410193  +0.187054   0.513889
mistral_7b_instruct_v0_2           0.346790   0.465645  +0.118855   0.880952
llama31_8b_instruct                0.290288   0.538142  +0.247854   0.821429


Mistral uses a non-independent Mistral self-judge. FLAN's metric increase is qualified by its instruction-following and citation failures. Llama provides the strongest evaluator-independent evidence that RAG helped on this small benchmark; this is not a general model-ranking claim.

In [3]:
print(f"{'Model':32} {'RAG ms':>10} {'Chunk rows':>12} {'Exact cite':>11} {'Echoes':>8}")
for model, data in summary['models'].items():
    rag = data['uninspected']['conditions']['rag']
    print(f"{model:32} {rag['mean_generation_latency_ms']:10.2f} "
          f"{rag['rows_with_eligible_chunk_mention']:>8}/7 "
          f"{rag['citation_format_compliant_rows']:>7}/7 "
          f"{rag['question_echoes']:8d}")

Model                                RAG ms   Chunk rows  Exact cite   Echoes
flan_t5_base                         396.36        0/7       0/7        1
mistral_7b_instruct_v0_2            6333.67        5/7       1/7        0
llama31_8b_instruct                 6296.25        6/7       1/7        0


## Retrieval ablation conclusion

- Top-k 1: fact recall 0.7000; with reranking, 0.7417.
- Top-k 3 and 5: fact recall 1.0000; top-k 5 adds context without recall benefit.
- Chunk sizes 256/512/1024 were identical because all governed sections were shorter than 256 tokens.
- Retained smoke default: top-k 3 without reranking.

The June internship sources were validated only in a separate private Chroma collection. Private content and raw outputs are not loaded by this notebook.